# OOV summary: word2vec vs. GloVe vs. fastText

Runs the three eval notebooks one after the other with the shared `preprocess()` from `preprocessing.py` and compares their metrics. Workflow: edit `preprocessing.py`, run this notebook.

In [1]:
import importlib
import inspect
import json
from pathlib import Path

import nbformat
import pandas as pd
from nbclient import NotebookClient

import preprocessing

importlib.reload(preprocessing)  # pick up the latest edit
print(inspect.getsource(preprocessing.preprocess))

def preprocess(text):
    tokens = nltk.word_tokenize(text)
    return [token for token in tokens if any(character.isalnum() for character in token)]



In [2]:
EVALS = {"w2v": "w2v_oov.ipynb", "glove": "glove_oov.ipynb", "fasttext": "fasttext_oov.ipynb"}

# one kernel per notebook, one after the other: word2vec alone needs about 4 GB, so never load all three at once
for notebook_path in EVALS.values():
    notebook = nbformat.read(notebook_path, as_version=4)
    NotebookClient(notebook, kernel_name="python3-cuda", timeout=1800, resources={"metadata": {"path": "."}}).execute()
    nbformat.write(notebook, notebook_path)  # the executed outputs stay visible in each notebook
    print(f"{notebook_path} done")

[IPKernelApp] WARNING | Kernel is running over TCP without encryption. All communication (including code and outputs) is sent in plain text and is susceptible to eavesdropping. Use IPC transport or launch with kernel manager-provisioned CurveZMQ keys to enable transport encryption.


w2v_oov.ipynb done


[IPKernelApp] WARNING | Kernel is running over TCP without encryption. All communication (including code and outputs) is sent in plain text and is susceptible to eavesdropping. Use IPC transport or launch with kernel manager-provisioned CurveZMQ keys to enable transport encryption.


glove_oov.ipynb done


[IPKernelApp] WARNING | Kernel is running over TCP without encryption. All communication (including code and outputs) is sent in plain text and is susceptible to eavesdropping. Use IPC transport or launch with kernel manager-provisioned CurveZMQ keys to enable transport encryption.


fasttext_oov.ipynb done


In [3]:
results = {name: json.loads(Path(f"results/{name}.json").read_text()) for name in EVALS}

table = pd.DataFrame({name: result["metrics"] for name, result in results.items()})
table = table.loc[["vocab_size", "hits_exact", "hits_lowercase", "occurrences", "rate", "unique_types", "type_rate", "empty_sentences", "tokens", "types"]]
table.index = ["vocabulary", "hits exact", "hits lowercase", "OOV occurrences", "OOV rate", "OOV types", "OOV type rate", "empty sentences", "tokens", "types"]
table.loc[["OOV rate", "OOV type rate"]] = table.loc[["OOV rate", "OOV type rate"]].map(lambda value: f"{value:.2%}")
table

/tmp/ipykernel_156161/2364176614.py:6: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '['11.74%' '6.00%']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  table.loc[["OOV rate", "OOV type rate"]] = table.loc[["OOV rate", "OOV type rate"]].map(lambda value: f"{value:.2%}")
/tmp/ipykernel_156161/2364176614.py:6: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '['0.17%' '4.52%']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  table.loc[["OOV rate", "OOV type rate"]] = table.loc[["OOV rate", "OOV type rate"]].map(lambda value: f"{value:.2%}")
/tmp/ipykernel_156161/2364176614.py:6: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '['0.67%' '3.11%']' has dtype incompatib

,w2v,glove,fasttext
vocabulary,3000000.0,400000.0,999999.0
hits exact,591672.0,575426.0,665907.0
hits lowercase,18.0,93825.0,26.0
OOV occurrences,78705.0,1144.0,4462.0
OOV rate,11.74%,0.17%,0.67%
OOV types,1142.0,861.0,593.0
OOV type rate,6.00%,4.52%,3.11%
empty sentences,6.0,7.0,7.0
tokens,670395.0,670395.0,670395.0
types,19041.0,19041.0,19041.0


In [4]:
for name, result in results.items():
    print(f"{name}: {result['top_oov'][:10]}")

w2v: [['to', 26514], ['a', 22768], ['and', 15023], ['of', 6595], ["'s", 5582], ['20', 75], ['10', 72], ['didnt', 58], ['cancelled', 48], ['100', 44]]
glove: [['selfie', 11], ['mTurk', 8], ['Zaiden', 7], ['mturk', 6], ['Mturk', 6], ['selfies', 6], ['A+', 6], ['Antonela', 5], ['Fariah', 5], ['dogwhistle', 5]]
fasttext: [["n't", 3702], ['mTurk', 8], ["o'clock", 7], ['Zaiden', 7], ['A+', 6], ['Mturk', 6], ['mturk', 6], ['Antonela', 5], ['Fariah', 5], ['saded', 4]]
